In [ ]:
#import sklearn
import igraph
import harmony
import anndata as ad
#import scanpy as sc
import pegasus as pg
import pandas as pd
import os
from scipy import sparse, io 
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
import seaborn as sb
import matplotlib
import matplotlib.colors as col
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import numpy as np
cmap2 = col.LinearSegmentedColormap.from_list('own',['#A0A0A0','#000000','red','orange'])

In [ ]:
from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42

In [ ]:
data = pg.read_input("/cluster3/yflu/STS/TMA/adata_TMA_merged.h5ad")

In [ ]:
pg.write_output(data,"/cluster3/yflu/STS/TMA/adata_TMA_merged_new.h5ad")

In [ ]:
data = pg.read_input("/cluster3/yflu/STS/TMA/adata_TMA_merged_new.h5ad")

In [ ]:
data

In [ ]:
pg.identify_robust_genes(data)
pg.log_norm(data,norm_count = 5000, base_matrix="raw.X")
pg.calc_signature_score(data, 'cell_cycle_human')
pg.pca(data)
pg.regress_out(data, attrs=['G1/S', 'G2/M'])
pg.neighbors(data,rep="pca_regressed")
pg.louvain(data,rep="pca_regressed",class_label='louvain_labels',resolution=0.8)
pg.umap(data,rep="pca_regressed",out_basis='umap')
pg.tsne(data,rep="pca_regressed",out_basis='tsne')

In [ ]:
pg.write_output(data,"/cluster3/yflu/STS/TMA/adata_TMA_merged_260114.h5ad")

In [ ]:
data

In [ ]:
pg.scatter(data, ["celltype"])

In [ ]:
data

In [ ]:
pg.tsne(data,rep="pca_regressed",out_basis='tsne')

In [ ]:
data = pg.aggregate_matrices('/cluster3/yflu/STS/TMA/TMA_sample_select_250728.csv')
data = data[:, data.var.index[0:5001]].copy()
pg.qc_metrics(data,min_genes=15,max_genes=3000,min_umis=40)
df_qc = pg.get_filter_stats(data)
pg.filter_data(data)

In [ ]:
help(pg.qc_metrics)

In [ ]:
cell_merged_STS = pd.read_csv("/cluster3/yflu/STS/TMA/cell_merged_STS.csv",sep=',',header=0)
data=data[data.obs.index.isin(cell_merged_STS["Cell_ID_new"])].copy()

In [ ]:
cell_mapping = pd.DataFrame({
    'order': data.obs.index.to_list(),
})
sort_mapping = cell_mapping.reset_index().set_index('order')

In [ ]:
cell_merged_STS = cell_merged_STS[cell_merged_STS["Cell_ID_new"].isin(data.obs.index)]
cell_merged_STS['cell_order'] = cell_merged_STS['Cell_ID_new'].map(sort_mapping['index'])
cell_merged_STS = cell_merged_STS.sort_values('cell_order')

In [ ]:
cell_merged_STS.index = cell_merged_STS["Cell_ID_new"]
data.obs["Disease"] = pd.Series(cell_merged_STS["Disease"], dtype="category")
data.obs["Sample"] = pd.Series(cell_merged_STS["Sample"], dtype="category")

In [ ]:
pg.write_output(data,'/cluster3/yflu/STS/TMA/STS_TMA_counts_20250730.h5ad')

In [ ]:
data = pg.read_input('/cluster3/yflu/STS/TMA/STS_TMA_counts_20250730.h5ad')

In [ ]:
samples = data.obs["Sample"].unique()

In [ ]:
samples

In [ ]:
samples = pd.Series(samples).loc[lambda x: x != "T491"].astype("category")
samples =pd.Series(samples, dtype="category").cat.remove_categories(["T491"])

In [ ]:
samples = ['T640T2','T877','T937','T1057','T976']

In [ ]:
samples[0]

In [ ]:
for sample in samples:
    data_subset = data[data.obs['Sample'] == sample].copy()
    file_name = f"/cluster3/yflu/STS/TMA/separated/{sample}_counts.h5ad"  # Customize the filename as needed
    pg.write_output(data_subset,file_name)
    print(f"Subsetting for sample '{sample}' is done and saved to {file_name}.")

In [ ]:
for sample in samples:
    data_subset = data[data.obs['Sample'] == sample].copy()
    pg.identify_robust_genes(data_subset)
    pg.log_norm(data_subset,norm_count = 5000)
    pg.calc_signature_score(data_subset, 'cell_cycle_human')
    pg.pca(data_subset)
    pg.regress_out(data_subset, attrs=['G1/S', 'G2/M'])
    pg.neighbors(data_subset,rep="pca_regressed")
    pg.louvain(data_subset,rep="pca_regressed",class_label='louvain_labels',resolution=0.8)
    pg.umap(data_subset,rep="pca_regressed",out_basis='umap')
    pg.tsne(data_subset,rep="pca_regressed",out_basis='tsne')
    # Save the subset as an .h5ad file
    file_name = f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad"  # Customize the filename as needed
    pg.write_output(data_subset,file_name)
    print(f"Subsetting for sample '{sample}' is done and saved to {file_name}.")

In [ ]:
sample = "T1091"
data = pg.read_input(f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad")
pg.louvain(data,rep="pca_regressed",class_label='louvain_labels',resolution=1.0)
pg.scatter(data, ["louvain_labels"])

In [ ]:
pg.write_output(data,f"/cluster3/yflu/STS/TMA/separated/{sample}_subset.h5ad")

In [ ]:
pg.write_output(data,f"/cluster3/yflu/STS/TMA/separated/{sample}/{sample}_preprocessed.h5ad")